#### Scrape videos from 5 hashtags(comedy, cooking, finance, fitness, study, tech)

In [5]:
import pandas as pd
import os

# Filter condition 
fans_min, fans_max = 3000, 30000
video_min = 30

# File list
file_list = ['comedy.csv', 'cooking.csv', 'finance.csv', 'fitness.csv', 'study.csv', 'tech.csv']

username_col = 'authorMeta.name'  

print(f"Result (#fans {fans_min}-{fans_max}, #videos > {video_min}):\n")

for file in file_list:
    if os.path.exists(file):
        df = pd.read_csv(file)

        condition = (
            (df['authorMeta.fans'] >= fans_min) &
            (df['authorMeta.fans'] <= fans_max) &
            (df['authorMeta.video'] > video_min)
        )

        filtered = df[condition].copy()

        filtered = filtered.dropna(subset=[username_col])
        filtered = filtered[filtered[username_col].astype(str).str.strip() != ""]

        unique_users = filtered.drop_duplicates(subset=[username_col])

        count = len(unique_users)

        print(f"{file}: {count} users")

    else:
        print(f"{file}: File not exist")

Result (#fans 3000-30000, #videos > 30):

comedy.csv: 101 users
cooking.csv: 78 users
finance.csv: 127 users
fitness.csv: 107 users
study.csv: 133 users
tech.csv: 116 users


##### Select ~80 accouts for each category

In [3]:
max_users_per_category = 80

# Possible username columns, checked in order
username_candidates = [
    'authorMeta.name',
    'authorMeta.nickName',
    'authorMeta.nickname',
    'authorMeta.uniqueId',
    'authorMeta.id'
]

selected_users = []

print(f"Result (fans: {fans_min}-{fans_max}, videos > {video_min})\n")

for file in file_list:
    if not os.path.exists(file):
        print(f"{file}: File not exist")
        continue

    df = pd.read_csv(file)

    # Check required columns
    required_cols = ['authorMeta.fans', 'authorMeta.video']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"{file}: Missing columns {missing_cols}")
        continue

    # Find username column
    username_col = None
    for col in username_candidates:
        if col in df.columns:
            username_col = col
            break

    if username_col is None:
        print(f"{file}: No username column found")
        continue

    # Filter rows
    filtered = df[
        (df['authorMeta.fans'] >= fans_min) &
        (df['authorMeta.fans'] <= fans_max) &
        (df['authorMeta.video'] > video_min)
    ].copy()

    # Keep only needed columns
    filtered = filtered[[username_col, 'authorMeta.fans']].copy()
    filtered.columns = ['username', 'fans']

    # Remove empty usernames
    filtered = filtered.dropna(subset=['username'])
    filtered = filtered[filtered['username'].astype(str).str.strip() != ""]

    # Deduplicate users within the category
    filtered = filtered.drop_duplicates(subset=['username'])

    # Select up to 80 users
    selected = filtered.head(max_users_per_category).copy()

    # Optional: keep category name for checking
    selected['category'] = os.path.splitext(file)[0]

    selected_users.append(selected)

    print(f"{file}: {len(filtered)} qualified users, selected {len(selected)}")

# Merge all categories
if selected_users:
    final_df = pd.concat(selected_users, ignore_index=True)

    # If you want to ensure no duplicate usernames across all categories, uncomment:
    # final_df = final_df.drop_duplicates(subset=['username'])

    # Save only username and fans
    output_df = final_df[['username', 'fans']]
    output_df.to_csv('selected_users.csv', index=False, encoding='utf-8-sig')

    print("\nSaved to selected_users.csv")
    print(f"Total selected users: {len(output_df)}")
else:
    print("\nNo users selected.")

Result (fans: 3000-30000, videos > 30)

comedy.csv: 101 qualified users, selected 80
cooking.csv: 78 qualified users, selected 78
finance.csv: 127 qualified users, selected 80
fitness.csv: 107 qualified users, selected 80
study.csv: 133 qualified users, selected 80
tech.csv: 116 qualified users, selected 80

Saved to selected_users.csv
Total selected users: 478
